# CLSA fundus → RETFound smoke test

This notebook performs an end-to-end smoke test on four baseline participants (both eyes, eight images):

1. extract encrypted JPEGs from the baseline archive;
2. create a governed image manifest;
3. crop the retinal field and normalize each image to 256×256;
4. construct the standardized 224×224 RETFound tensor input;
5. run RETFound on CUDA and save eight 1,024-dimensional vectors.

The archive password and Hugging Face token are temporary text widgets. They are not written to output files, but the widgets are not masked. Do not share the screen, print their values, or commit populated credentials.

## One-time environment setup

Run the next cell on a GPU cluster. It deliberately does not install PyTorch, preserving the CUDA-enabled PyTorch supplied by Databricks Runtime ML. After Python restarts, continue from the CUDA check.

In [ ]:
%pip install -r /Workspace/Users/ad0038@pennmedicine.upenn.edu/CLSA/CLSA_retina/requirements-retfound.txt
dbutils.library.restartPython()

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Attach this notebook to GPU compute."
print("GPU:", torch.cuda.get_device_name(0))

## Configuration

Paths and reproducible settings are fixed in the configuration cell. Enter only the temporary archive password and, if a gated download is required, a rotated Hugging Face token.

In [ ]:
dbutils.widgets.text("archive_password", "", "Archive password (temporary)")
dbutils.widgets.text("hf_token", "", "Hugging Face token (temporary)")

In [ ]:
import os
from pathlib import Path
import sys

repo_root = "/Workspace/Users/ad0038@pennmedicine.upenn.edu/CLSA/CLSA_retina"
archive_path = "/Volumes/ophthalmology_analytics/dev_optic/clsa_dataset/2209017_BL.zip"
output_root = Path("/Volumes/ophthalmology_analytics/dev_optic/clsa_dataset/derived/clsa_retinal_aging/fundus_retfound_smoke")
n_participants = 4
batch_size = 2
force_embeddings = True
retfound_repo = None
checkpoint_path = None

module_path = Path(repo_root) / "src" / "fundus_retfound_pipeline.py"
assert module_path.exists(), f"Pipeline module was not found: {module_path}"
if str(module_path.parent) not in sys.path:
    sys.path.insert(0, str(module_path.parent))

from fundus_retfound_pipeline import (
    QualityConfig,
    RETFoundConfig,
    extract_retfound_embeddings,
    load_retfound_model,
    prepare_model_input,
    run_quality_pipeline,
)

print("Pipeline imports succeeded")
print("Output root:", output_root)

## 1. Extract four participants × two eyes

The release layout is `2209017_BL/<participant_id>/retinal_<left|right>.jpeg`. The code selects complete left/right pairs and skips the few entries marked as possible WinZip AES because Python's standard ZIP reader cannot decrypt those entries.

In [ ]:
from collections import defaultdict
import re
import shutil
import zipfile

archive_password = dbutils.widgets.get("archive_password")
if not archive_password:
    raise ValueError("Enter the archive password in the top widget.")

archive_stem = Path(archive_path).stem
visit = archive_stem.rsplit("_", 1)[-1].upper()
member_pattern = re.compile(
    rf"^{re.escape(archive_stem)}/"
    r"(?P<participant_id>\d+)/"
    r"retinal_(?P<eye>left|right)\.jpeg$",
    re.IGNORECASE,
)

input_root = output_root / "00_input" / visit
input_root.mkdir(parents=True, exist_ok=True)
password_bytes = archive_password.encode("utf-8")
manifest_rows = []

try:
    with zipfile.ZipFile(archive_path) as archive:
        images_by_participant = defaultdict(dict)
        for info in archive.infolist():
            if info.is_dir() or b"\x01\x99" in info.extra:
                continue
            match = member_pattern.match(info.filename)
            if match:
                images_by_participant[match.group("participant_id")][
                    match.group("eye").lower()
                ] = info

        eligible = sorted(
            participant_id
            for participant_id, eyes in images_by_participant.items()
            if {"left", "right"}.issubset(eyes)
        )
        selected = eligible[:n_participants]
        if len(selected) != n_participants:
            raise RuntimeError(
                f"Requested {n_participants} complete pairs; found {len(selected)}."
            )

        for participant_id in selected:
            for eye_name in ("left", "right"):
                info = images_by_participant[participant_id][eye_name]
                participant_dir = input_root / participant_id
                participant_dir.mkdir(parents=True, exist_ok=True)
                destination = participant_dir / f"retinal_{eye_name}.jpeg"

                if (
                    not destination.exists()
                    or destination.stat().st_size != info.file_size
                ):
                    with archive.open(info, pwd=password_bytes) as source:
                        with destination.open("wb") as target:
                            shutil.copyfileobj(
                                source, target, length=8 * 1024 * 1024
                            )

                eye = "L" if eye_name == "left" else "R"
                manifest_rows.append(
                    {
                        "participant_id": participant_id,
                        "image_path": str(destination),
                        "eye": eye,
                        "eye_parsed": eye,
                        "visit": visit,
                        "filename": destination.name,
                        "archive_path": archive_path,
                        "archive_member": info.filename,
                        "bytes": int(info.file_size),
                    }
                )
finally:
    password_bytes = b""
    archive_password = ""

expected_images = 2 * n_participants
assert len(manifest_rows) == expected_images
manifest_spark = spark.createDataFrame(manifest_rows)
manifest_delta_path = str(output_root / "00_input" / "fundus_smoke_manifest")
(
    manifest_spark.write.format("delta")
    .mode("overwrite")
    .save(manifest_delta_path)
)
manifest = manifest_spark.toPandas()
print(f"Extracted and indexed {len(manifest):,} images")
display(manifest_spark.orderBy("participant_id", "eye"))

## 2. Confirm the original images

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

figure, axes = plt.subplots(
    nrows=n_participants,
    ncols=2,
    figsize=(10, 4 * n_participants),
)
for image_number, (axis, record) in enumerate(
    zip(np.atleast_1d(axes).flat, manifest_rows), start=1
):
    with Image.open(record["image_path"]) as image:
        image.load()
        axis.imshow(image)
        axis.set_title(f"Image {image_number} — eye {record['eye']}")
    axis.axis("off")
plt.tight_layout()
plt.show()

## 3. Retinal-field crop, square padding, and 256×256 quality image

Technical QC uses the retinal foreground to crop dark borders, fills non-retinal background with the median retinal color, pads to a square, and resizes with bicubic interpolation to 256×256. Quality failures remain in the manifest and are never silently deleted.

In [ ]:
quality_config = QualityConfig(
    output_size=256,
    model_input_size=224,
    save_preprocessed=True,
)
quality = run_quality_pipeline(
    manifest,
    output_root / "01_quality",
    quality_config,
)

display(
    quality[
        [
            "participant_id",
            "eye",
            "quality_pass",
            "quality_reasons",
            "original_width",
            "original_height",
            "retina_fraction",
            "brightness_mean",
            "contrast_std",
            "gradient_energy",
            "processed_image_path",
        ]
    ]
)

failed = quality.loc[~quality["quality_pass"].fillna(False)]
if not failed.empty:
    raise RuntimeError(
        "The smoke test requires all eight images to pass technical QC. "
        "Review the displayed failure reasons before changing thresholds."
    )
assert len(quality) == 2 * n_participants
print("All smoke-test images passed technical QC")

In [ ]:
processed = quality["processed_image_path"].dropna().tolist()
figure, axes = plt.subplots(
    nrows=n_participants,
    ncols=2,
    figsize=(10, 4 * n_participants),
)
for image_number, (axis, processed_path) in enumerate(
    zip(np.atleast_1d(axes).flat, processed), start=1
):
    with Image.open(processed_path) as image:
        assert image.size == (256, 256)
        axis.imshow(image)
        axis.set_title(f"Cropped 256×256 — image {image_number}")
    axis.axis("off")
plt.tight_layout()
plt.show()

## 4. Verify RETFound model input

RETFound receives a 224×224 RGB array. Each image channel is standardized independently to approximately zero mean and unit standard deviation.

In [ ]:
sample_input = prepare_model_input(
    quality.iloc[0]["image_path"],
    quality_config,
)
assert sample_input.shape == (224, 224, 3)
assert np.isfinite(sample_input).all()
print("RETFound input shape:", sample_input.shape)
print("Channel means:", sample_input.mean(axis=(0, 1)))
print("Channel standard deviations:", sample_input.std(axis=(0, 1)))

## 5. Load RETFound on CUDA

Leave the repository and checkpoint widgets blank to clone the official repository and download the gated checkpoint. The token is removed from the process environment immediately after the model loads.

In [ ]:
temporary_hf_token = dbutils.widgets.get("hf_token").strip()
if not checkpoint_path and not temporary_hf_token:
    raise ValueError("Enter the Hugging Face token in the top widget.")

retfound_config = RETFoundConfig(
    repo_path=retfound_repo,
    checkpoint_path=checkpoint_path,
    allow_downloads=True,
    device="cuda",
    batch_size=batch_size,
)

if temporary_hf_token:
    os.environ["HF_TOKEN"] = temporary_hf_token
try:
    model, device, resolved_repo, resolved_checkpoint = load_retfound_model(
        retfound_config
    )
finally:
    os.environ.pop("HF_TOKEN", None)
    temporary_hf_token = ""

assert device == "cuda"
print("Device:", device)
print("RETFound repository:", resolved_repo)
print("Checkpoint:", resolved_checkpoint)

## 6. Generate and validate the final vectors

In [ ]:
embeddings = extract_retfound_embeddings(
    quality,
    output_root / "02_embeddings",
    retfound_config,
    quality_config,
    model=model,
    device=device,
    checkpoint_path=resolved_checkpoint,
    force=force_embeddings,
)

embedding_matrix = np.stack(embeddings["embedding"].to_numpy()).astype(
    np.float32
)
expected_images = 2 * n_participants
assert embedding_matrix.shape == (expected_images, 1024), (
    f"Expected {(expected_images, 1024)}, got {embedding_matrix.shape}"
)
assert np.isfinite(embedding_matrix).all(), "Vectors contain NaN or infinity."

norms = np.linalg.norm(embedding_matrix, axis=1)
assert np.all(norms > 0), "At least one vector has zero length."
print("Final embedding matrix shape:", embedding_matrix.shape)
print("Vector dtype:", embedding_matrix.dtype)
print("Vector L2 norm range:", float(norms.min()), float(norms.max()))

In [ ]:
vector_records = []
for row_number, (_, record) in enumerate(embeddings.iterrows()):
    vector_records.append(
        {
            "participant_id": str(record["participant_id"]),
            "visit": str(record.get("visit", visit)),
            "eye": str(record.get("eye", "")),
            "image_path": str(record["image_path"]),
            "embedding_dim": int(record["embedding_dim"]),
            "retfound_model": str(record["retfound_model"]),
            "retfound_checkpoint_sha256": str(
                record["retfound_checkpoint_sha256"]
            ),
            "embedding": [float(value) for value in embedding_matrix[row_number]],
        }
    )

vectors_spark = spark.createDataFrame(vector_records)
vectors_delta_path = str(
    output_root / "02_embeddings" / "retfound_embeddings_delta"
)
(
    vectors_spark.write.format("delta")
    .mode("overwrite")
    .save(vectors_delta_path)
)

preview_rows = []
for row_number, record in enumerate(vector_records):
    preview_rows.append(
        {
            "participant_id": record["participant_id"],
            "visit": record["visit"],
            "eye": record["eye"],
            "embedding_dim": record["embedding_dim"],
            "l2_norm": float(norms[row_number]),
            "first_8_values": record["embedding"][:8],
        }
    )
display(spark.createDataFrame(preview_rows))

print("Parquet vectors:", output_root / "02_embeddings/retfound_embeddings.parquet")
print("Delta vectors:", vectors_delta_path)

## 7. Remove temporary credential widgets

Run this only after extraction and checkpoint loading have completed. Re-running the smoke test later requires re-running the widget-definition cell and entering the values again.

In [ ]:
os.environ.pop("HF_TOKEN", None)
for widget_name in ("archive_password", "hf_token"):
    try:
        dbutils.widgets.remove(widget_name)
    except Exception:
        pass
print("Temporary credential widgets removed")

## Expected durable outputs

- `00_input/fundus_smoke_manifest`: Delta manifest for the eight extracted images.
- `01_quality/preprocessed_256/`: cropped and normalized 256×256 JPEGs.
- `01_quality/fundus_quality_manifest.parquet`: image-level quality metrics and crop coordinates.
- `02_embeddings/retfound_embeddings.parquet`: canonical pipeline vectors.
- `02_embeddings/retfound_embeddings_delta`: Databricks-friendly Delta vector table.
- `02_embeddings/retfound_embedding_metadata.json`: checkpoint hash, model configuration, and counts.

A successful smoke test ends with an embedding matrix shape of `(8, 1024)`.